# 3 en Raya (Accion - Valor)

In [1]:
import numpy as np


class Board():
    def __init__(self):
        self.state = np.zeros((3,3))

    def valid_moves(self):
        return [(i, j) for j in range(3) for i in range(3) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] == 0:
            self.state[row, col] = symbol
        else:
            raise ValueError ("movimiento ilegal !")

    def is_game_over(self):
        # comprobar filas y columnas
        if (self.state.sum(axis=0) == 3).sum() >= 1 or (self.state.sum(axis=1) == 3).sum() >= 1:
            return 1
        if (self.state.sum(axis=0) == -3).sum() >= 1 or (self.state.sum(axis=1) == -3).sum() >= 1:
            return -1
        # comprobar diagonales
        diag_sums = [
            sum([self.state[i, i] for i in range(3)]),
            sum([self.state[i, 3 - i - 1] for i in range(3)]),
        ]
        if diag_sums[0] == 3 or diag_sums[1] == 3:
            return 1
        if diag_sums[0] == -3 or diag_sums[1] == -3:
            return -1
        # empate
        if len(self.valid_moves()) == 0:
            return 0
        # seguir jugando
        return None

    def reset(self):
        self.state = np.zeros((3,3))


In [2]:
from tqdm import tqdm

class Game():
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()

            # Orden aleatorio cada partida
            players_this_round = self.players[:]
            np.random.shuffle(players_this_round)

            game_over = False
            while not game_over:
                for player in players_this_round:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins

    def reward(self, reward):
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 1.0  # <-- optimista en vez de 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

## Valores Optimistas

In [5]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {} # tabla con pares estado -> valor
        self.alpha = alpha         # learning rate
        self.positions = []       # guardamos todas las posiciones de la partida
        self.prob_exp = prob_exp   # probabilidad de explorar

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        # exploracion
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            # vamos a una posición aleatoria
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        # explotacion
        # vamos a la posición con más valor
        max_value = -1000
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(3*3))
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(3*3)))


    def reward(self, reward):
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 1.0  # <-- optimista en vez de 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

## Gradiente

In [6]:
class AgentGradiente():
    def __init__(self, alpha=0.1):
        self.preferences = {}   # H(estado) -> preferencia
        self.alpha = alpha
        self.positions = []
        self.recompensas = []   # historial para calcular la media

    def reset(self):
        self.positions = []
        self.recompensas = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()

        # obtener preferencias y convertir a probabilidades con softmax
        H = []
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(9))
            H.append(self.preferences.get(next_state, 0.0))

        H = np.array(H)
        exp_H = np.exp(H - H.max())      # estabilidad numérica
        pi = exp_H / exp_H.sum()         # softmax -> probabilidades

        # elegir acción según probabilidades
        ix = np.random.choice(len(valid_moves), p=pi)
        return valid_moves[ix]

    def update(self, board):
        self.positions.append(str(board.state.reshape(9)))

    def reward(self, reward):
        self.recompensas.append(reward)
        recompensa_media = np.mean(self.recompensas)

        # reconstruir preferencias y probabilidades de los estados visitados
        H = np.array([self.preferences.get(p, 0.0) for p in self.positions])
        exp_H = np.exp(H - H.max())
        pi = exp_H / exp_H.sum()

        for ix, p in enumerate(self.positions):
            if p not in self.preferences:
                self.preferences[p] = 0.0
            # estado escogido -> sube preferencia si recompensa > media
            # otros estados -> bajan preferencia
            self.preferences[p] += self.alpha * (reward - recompensa_media) * (1 - pi[ix])

In [ ]:
agent1 = AgentGradiente(alpha=0.1)
agent2 = Agent()
game = Game(agent1, agent2)
game.selfplay(300000)